In [1]:
#!/usr/bin/env python3
"""
STEP 2 — TRAINING (COLLEGE DEMO — MEMORIZATION MODE)
======================================================
BiLSTM + Attention — trained on ALL videos.
No dropout, no regularization — pure memorization.
Target: 99-100% accuracy.
All files use 'demo_' prefix.
"""

# ==============================================================
# SECTION 1 — GPU SETUP (RUN THIS CELL FIRST AFTER KERNEL RESTART)
# ==============================================================
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print(f'GPUs: {len(gpus)}')
if gpus:
    tf.config.experimental.set_memory_growth(gpus[0], True)
print(f'TensorFlow: {tf.__version__}')
print('GPU ready ✅')

2026-03-07 03:54:28.984678: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


GPUs: 1
TensorFlow: 2.16.1
GPU ready ✅


In [2]:
# ==============================================================
# SECTION 2 — CONFIG
# ==============================================================
SAVE_DIR    = '/workspace/deepfake/preprocessed'
PREFIX      = 'demo_'
MODEL_PATH  = os.path.join(SAVE_DIR, f'{PREFIX}bilstm.keras')
ATTN_PATH   = os.path.join(SAVE_DIR, f'{PREFIX}attention.keras')
FEAT_PATH   = os.path.join(SAVE_DIR, f'{PREFIX}features.npy')
PRED_PATH   = os.path.join(SAVE_DIR, f'{PREFIX}predictions.npy')
LOG_PATH    = os.path.join(SAVE_DIR, f'{PREFIX}log.csv')
PLOT_PATH   = os.path.join(SAVE_DIR, f'{PREFIX}curves.png')

IMG_SIZE    = 224
MAX_FRAMES  = 20

BATCH_SIZE  = 16
EPOCHS      = 100
LR          = 5e-4
SEED        = 42

print('=' * 60)
print('COLLEGE DEMO — MEMORIZATION MODE')
print(f'All files: {PREFIX}*')
print(f'Model will be saved to: {MODEL_PATH}')
print('=' * 60)

COLLEGE DEMO — MEMORIZATION MODE
All files: demo_*
Model will be saved to: /workspace/deepfake/preprocessed/demo_bilstm.keras


In [3]:
# ==============================================================
# SECTION 3 — IMPORTS
# ==============================================================
import numpy as np
from tensorflow.keras.applications import EfficientNetB4
from tensorflow.keras.layers import (Input, Dense,
    BatchNormalization, Bidirectional, LSTM, Layer)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import (ModelCheckpoint, CSVLogger,
    LearningRateScheduler)
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.utils.class_weight import compute_class_weight
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import gc, time

np.random.seed(SEED)
tf.random.set_seed(SEED)
print('Imports done ✅')

Imports done ✅


In [4]:
# ==============================================================
# SECTION 4 — LOAD DATA + EXTRACT CNN FEATURES
# ==============================================================
print('\nLoading demo data...')
demo_X = np.load(os.path.join(SAVE_DIR, f'{PREFIX}X.npy'))
demo_y = np.load(os.path.join(SAVE_DIR, f'{PREFIX}y.npy'))

n_videos = demo_X.shape[0]
print(f'Shape  : {demo_X.shape}')
print(f'Videos : {n_videos}')
print(f'Real   : {(demo_y==0).sum()}')
print(f'Fake   : {(demo_y==1).sum()}')

# Class weights
cls = np.unique(demo_y)
wts = compute_class_weight('balanced', classes=cls, y=demo_y)
demo_class_weight = {int(c): float(w) for c, w in zip(cls, wts)}
print(f'Class weights: {demo_class_weight}')

# CNN feature extraction
print('\nLoading EfficientNet-B4 for feature extraction...')
cnn_backbone = EfficientNetB4(
    weights='imagenet', include_top=False, pooling='avg',
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)
cnn_backbone.trainable = False
FEAT_DIM = cnn_backbone.output_shape[-1]
print(f'Feature dim: {FEAT_DIM}')

if os.path.exists(FEAT_PATH):
    demo_feats = np.load(FEAT_PATH)
    print(f'Loaded features from cache: {demo_feats.shape}')
else:
    print(f'Extracting features from {n_videos} videos...')
    t0 = time.time()
    demo_feats = np.zeros((n_videos, MAX_FRAMES, FEAT_DIM), dtype=np.float32)
    for i in range(n_videos):
        frames_i = demo_X[i]  # (20, 224, 224, 3)
        feats_i = cnn_backbone.predict(frames_i, batch_size=20, verbose=0)  # (20, 1792)
        demo_feats[i] = feats_i
        if (i+1) % 50 == 0:
            print(f'  {i+1}/{n_videos}...')
    np.save(FEAT_PATH, demo_feats)
    elapsed = time.time() - t0
    print(f'Features extracted and saved: {demo_feats.shape} in {elapsed/60:.1f} min')

# Free raw frames — we only need features now
del demo_X
gc.collect()
print('Raw frames freed from memory')

# L2 normalize features
feat_norms = np.linalg.norm(demo_feats, axis=-1, keepdims=True) + 1e-8
demo_feats = demo_feats / feat_norms
print(f'demo_feats: {demo_feats.shape} (L2 normalized)')




Loading demo data...
Shape  : (732, 20, 224, 224, 3)
Videos : 732
Real   : 363
Fake   : 369
Class weights: {0: 1.0082644628099173, 1: 0.991869918699187}

Loading EfficientNet-B4 for feature extraction...


2026-03-07 03:55:08.466195: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1928] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 38380 MB memory:  -> device: 0, name: NVIDIA A100-SXM4-40GB, pci bus id: 0000:07:00.0, compute capability: 8.0


Feature dim: 1792
Extracting features from 732 videos...


I0000 00:00:1772855714.940827 3735139 service.cc:145] XLA service 0x7f0b18003d00 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1772855714.940889 3735139 service.cc:153]   StreamExecutor device (0): NVIDIA A100-SXM4-40GB, Compute Capability 8.0
2026-03-07 03:55:15.134194: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2026-03-07 03:55:15.910353: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:465] Loaded cuDNN version 8906
I0000 00:00:1772855720.868103 3735139 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


  50/732...
  100/732...
  150/732...
  200/732...
  250/732...
  300/732...
  350/732...
  400/732...
  450/732...
  500/732...
  550/732...
  600/732...
  650/732...
  700/732...
Features extracted and saved: (732, 20, 1792) in 1.7 min
Raw frames freed from memory
demo_feats: (732, 20, 1792) (L2 normalized)


In [5]:
# ==============================================================
# SECTION 5 — BiLSTM + ATTENTION MODEL (NO REGULARIZATION)
# ==============================================================
class DemoAttention(Layer):
    """Temporal attention for BiLSTM output."""
    def __init__(self, attn_units=64, **kwargs):
        super().__init__(**kwargs)
        self.attn_units = attn_units

    def build(self, input_shape):
        self.W_attn = Dense(self.attn_units, activation='tanh', name='demo_attn_tanh')
        self.v_attn = Dense(1, use_bias=False, name='demo_attn_score')
        super().build(input_shape)

    def call(self, lstm_out):
        # lstm_out: (batch, 20, hidden_dim)
        score = self.v_attn(self.W_attn(lstm_out))           # (batch, 20, 1)
        weights = tf.nn.softmax(score, axis=1)                # (batch, 20, 1)
        context = tf.reduce_sum(lstm_out * weights, axis=1)   # (batch, hidden_dim)
        return context, weights

    def get_config(self):
        cfg = super().get_config()
        cfg['attn_units'] = self.attn_units
        return cfg


def build_demo_model():
    demo_inp = Input(shape=(MAX_FRAMES, FEAT_DIM), name='demo_features')

    # Feature projection
    d = Dense(512, activation='relu', name='demo_proj')(demo_inp)
    d = BatchNormalization(name='demo_bn0')(d)

    # BiLSTM layer 1
    d = Bidirectional(LSTM(256, return_sequences=True, name='demo_lstm1'),
                      name='demo_bilstm1')(d)
    d = BatchNormalization(name='demo_bn1')(d)

    # BiLSTM layer 2
    d = Bidirectional(LSTM(128, return_sequences=True, name='demo_lstm2'),
                      name='demo_bilstm2')(d)
    d = BatchNormalization(name='demo_bn2')(d)

    # Temporal Attention
    demo_context, demo_attn_w = DemoAttention(64, name='demo_attn')(d)

    # Classification head — NO dropout, NO regularization
    d = BatchNormalization(name='demo_bn3')(demo_context)
    d = Dense(256, activation='relu', name='demo_fc1')(d)
    d = BatchNormalization(name='demo_bn4')(d)
    d = Dense(128, activation='relu', name='demo_fc2')(d)
    d = Dense(64, activation='relu', name='demo_fc3')(d)
    demo_out = Dense(1, activation='sigmoid', name='demo_pred')(d)

    # Main model: input → prediction
    demo_model = Model(demo_inp, demo_out, name='Demo_BiLSTM')

    # Attention model: input → [prediction, attention_weights]
    demo_attn_model = Model(demo_inp, [demo_out, demo_attn_w], name='Demo_Attn_Vis')

    return demo_model, demo_attn_model


demo_model, demo_attn_model = build_demo_model()

demo_model.compile(
    optimizer=Adam(learning_rate=LR),
    loss='binary_crossentropy',
    metrics=[
        'accuracy',
        tf.keras.metrics.AUC(name='auc'),
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall')
    ]
)

demo_model.summary(line_length=90)
total_params = demo_model.count_params()
print(f'\nTotal parameters: {total_params:,}')



Model: "Demo_BiLSTM"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                          ┃ Output Shape                 ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ demo_features (InputLayer)            │ (None, 20, 1792)             │               0 │
├───────────────────────────────────────┼──────────────────────────────┼─────────────────┤
│ demo_proj (Dense)                     │ (None, 20, 512)              │         918,016 │
├───────────────────────────────────────┼──────────────────────────────┼─────────────────┤
│ demo_bn0 (BatchNormalization)         │ (None, 20, 512)              │           2,048 │
├───────────────────────────────────────┼──────────────────────────────┼─────────────────┤
│ demo_bilstm1 (Bidirectional)          │ (None, 20, 512)              │       1,574,912 │
├───────────────────────────────────────┼──────────────────────────────┼─────────────────┤
│ demo_bn1 (BatchNormalization)         │ (None, 20, 512)              │           2,048 │
├───────────────────────────────────────┼──────────────────────────────┼─────────────────┤
│ demo_bilstm2 (Bidirectional)          │ (None, 20, 256)              │         656,384 │
├───────────────────────────────────────┼──────────────────────────────┼─────────────────┤
│ demo_bn2 (BatchNormalization)         │ (None, 20, 256)              │           1,024 │
├───────────────────────────────────────┼──────────────────────────────┼─────────────────┤
│ demo_attn (DemoAttention)             │ [(None, 256), (None, 20, 1)] │          16,512 │
├───────────────────────────────────────┼──────────────────────────────┼─────────────────┤
│ demo_bn3 (BatchNormalization)         │ (None, 256)                  │           1,024 │
├───────────────────────────────────────┼──────────────────────────────┼─────────────────┤
│ demo_fc1 (Dense)                      │ (None, 256)                  │          65,792 │
├───────────────────────────────────────┼──────────────────────────────┼─────────────────┤
│ demo_bn4 (BatchNormalization)         │ (None, 256)                  │           1,024 │
├───────────────────────────────────────┼──────────────────────────────┼─────────────────┤
│ demo_fc2 (Dense)                      │ (None, 128)                  │          32,896 │
├───────────────────────────────────────┼──────────────────────────────┼─────────────────┤
│ demo_fc3 (Dense)                      │ (None, 64)                   │           8,256 │
├───────────────────────────────────────┼──────────────────────────────┼─────────────────┤
│ demo_pred (Dense)                     │ (None, 1)                    │              65 │
└───────────────────────────────────────┴──────────────────────────────┴─────────────────┘

 Total params: 3,280,001 (12.51 MB)

 Trainable params: 3,276,417 (12.50 MB)

 Non-trainable params: 3,584 (14.00 KB)


Total parameters: 3,280,001


In [6]:
# ==============================================================
# SECTION 6 — TRAIN (MEMORIZE EVERYTHING)
# ==============================================================
demo_ds = tf.data.Dataset.from_tensor_slices(
    (demo_feats, demo_y.astype(np.float32))
)
demo_ds = demo_ds.shuffle(n_videos, seed=SEED).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

steps_per_epoch = n_videos // BATCH_SIZE
print(f'Steps per epoch: {steps_per_epoch}')


def demo_lr_schedule(epoch):
    """Warmup → constant → decay"""
    if epoch < 5:
        return LR * (epoch + 1) / 5     # warmup
    elif epoch < 50:
        return LR                         # constant
    elif epoch < 80:
        return LR * 0.5                   # half
    else:
        return LR * 0.1                   # low


demo_cbs = [
    ModelCheckpoint(MODEL_PATH, monitor='accuracy', mode='max',
                    save_best_only=True, verbose=1),
    LearningRateScheduler(demo_lr_schedule, verbose=0),
    CSVLogger(LOG_PATH)
]

print('\n' + '=' * 60)
print(f'TRAINING')
print(f'  Epochs       : {EPOCHS}')
print(f'  Videos       : {n_videos}')
print(f'  Batch size   : {BATCH_SIZE}')
print(f'  Learning rate: {LR}')
print(f'  Architecture : EfficientNet-B4 → BiLSTM×2 → Attention → Dense')
print(f'  Mode         : MEMORIZATION (no dropout, no regularization)')
print(f'  Save to      : {MODEL_PATH}')
print('=' * 60)

t_start = time.time()

demo_history = demo_model.fit(
    demo_ds,
    epochs=EPOCHS,
    callbacks=demo_cbs,
    class_weight=demo_class_weight,
    verbose=1
)

train_time = time.time() - t_start
print(f'\nTraining completed in {train_time/60:.1f} minutes')



Steps per epoch: 45

TRAINING
  Epochs       : 100
  Videos       : 732
  Batch size   : 16
  Learning rate: 0.0005
  Architecture : EfficientNet-B4 → BiLSTM×2 → Attention → Dense
  Mode         : MEMORIZATION (no dropout, no regularization)
  Save to      : /workspace/deepfake/preprocessed/demo_bilstm.keras
Epoch 1/100
45/46 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.7195 - auc: 0.7687 - loss: 0.5669 - precision: 0.6881 - recall: 0.8786
Epoch 1: accuracy improved from -inf to 0.80601, saving model to /workspace/deepfake/preprocessed/demo_bilstm.keras
46/46 ━━━━━━━━━━━━━━━━━━━━ 10s 17ms/step - accuracy: 0.7232 - auc: 0.7736 - loss: 0.5625 - precision: 0.6911 - recall: 0.8797 - learning_rate: 1.0000e-04
Epoch 2/100
45/46 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.9108 - auc: 0.9554 - loss: 0.2729 - precision: 0.8881 - recall: 0.9470
Epoch 2: accuracy improved from 0.80601 to 0.90574, saving model to /workspace/deepfake/preprocessed/demo_bilstm.keras
46/46 ━━━━━━━━━━━━━━━━━━━━ 

In [7]:
# ==============================================================
# SECTION 7 — VERIFY MEMORIZATION
# ==============================================================
print('\n' + '=' * 60)
print('MEMORIZATION CHECK')
print('=' * 60)

# Load best model
demo_model.load_weights(MODEL_PATH)
print(f'Loaded best model from: {MODEL_PATH}')

# Predict on ALL training data
demo_preds = demo_model.predict(demo_feats, batch_size=32, verbose=1).flatten()
demo_pred_labels = (demo_preds > 0.5).astype(int)

demo_acc = (demo_pred_labels == demo_y).mean()
demo_auc = roc_auc_score(demo_y, demo_preds)

print('\n' + classification_report(demo_y, demo_pred_labels,
                                    target_names=['REAL', 'FAKE'], digits=4))
print(f'Accuracy : {demo_acc * 100:.2f}%')
print(f'AUC      : {demo_auc:.4f}')

demo_wrong = np.where(demo_pred_labels != demo_y)[0]
print(f'Wrong predictions: {len(demo_wrong)} / {len(demo_y)}')

# If not perfect, train more with lower LR
if demo_acc < 0.99:
    demo_paths_check = np.load(os.path.join(SAVE_DIR, f'{PREFIX}paths.npy'))
    print('\nMisclassified videos:')
    for idx in demo_wrong[:10]:
        true_lbl = 'REAL' if demo_y[idx] == 0 else 'FAKE'
        print(f'  {os.path.basename(demo_paths_check[idx])}: true={true_lbl}, P(fake)={demo_preds[idx]:.4f}')

    print('\n⚠️  Not 99% yet — training 50 more epochs with LR × 0.1...')
    demo_model.compile(
        optimizer=Adam(learning_rate=LR * 0.1),
        loss='binary_crossentropy',
        metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
    )
    demo_model.fit(
        demo_ds, epochs=50,
        callbacks=[
            ModelCheckpoint(MODEL_PATH, monitor='accuracy',
                            mode='max', save_best_only=True, verbose=1),
            CSVLogger(os.path.join(SAVE_DIR, f'{PREFIX}log_extra1.csv'))
        ],
        class_weight=demo_class_weight,
        verbose=1
    )

    # Re-check
    demo_model.load_weights(MODEL_PATH)
    demo_preds = demo_model.predict(demo_feats, batch_size=32, verbose=0).flatten()
    demo_pred_labels = (demo_preds > 0.5).astype(int)
    demo_acc = (demo_pred_labels == demo_y).mean()
    demo_auc = roc_auc_score(demo_y, demo_preds)
    demo_wrong = np.where(demo_pred_labels != demo_y)[0]
    print(f'\nAfter extra training:')
    print(f'  Accuracy : {demo_acc * 100:.2f}%')
    print(f'  AUC      : {demo_auc:.4f}')
    print(f'  Wrong    : {len(demo_wrong)}')

# If STILL not perfect, one more round
if demo_acc < 0.99:
    print('\n⚠️  STILL not 99% — training 50 MORE epochs with LR × 0.01...')
    demo_model.compile(
        optimizer=Adam(learning_rate=LR * 0.01),
        loss='binary_crossentropy',
        metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
    )
    demo_model.fit(
        demo_ds, epochs=50,
        callbacks=[
            ModelCheckpoint(MODEL_PATH, monitor='accuracy',
                            mode='max', save_best_only=True, verbose=1),
            CSVLogger(os.path.join(SAVE_DIR, f'{PREFIX}log_extra2.csv'))
        ],
        class_weight=demo_class_weight,
        verbose=1
    )

    demo_model.load_weights(MODEL_PATH)
    demo_preds = demo_model.predict(demo_feats, batch_size=32, verbose=0).flatten()
    demo_pred_labels = (demo_preds > 0.5).astype(int)
    demo_acc = (demo_pred_labels == demo_y).mean()
    demo_auc = roc_auc_score(demo_y, demo_preds)
    demo_wrong = np.where(demo_pred_labels != demo_y)[0]
    print(f'\nFinal check:')
    print(f'  Accuracy : {demo_acc * 100:.2f}%')
    print(f'  Wrong    : {len(demo_wrong)}')




MEMORIZATION CHECK
Loaded best model from: /workspace/deepfake/preprocessed/demo_bilstm.keras
23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step

              precision    recall  f1-score   support

        REAL     1.0000    1.0000    1.0000       363
        FAKE     1.0000    1.0000    1.0000       369

    accuracy                         1.0000       732
   macro avg     1.0000    1.0000    1.0000       732
weighted avg     1.0000    1.0000    1.0000       732

Accuracy : 100.00%
AUC      : 1.0000
Wrong predictions: 0 / 732


In [8]:
# ==============================================================
# SECTION 8 — SAVE EVERYTHING
# ==============================================================
# Save attention model with same weights
demo_attn_model.set_weights(demo_model.get_weights())
demo_attn_model.save(ATTN_PATH)
print(f'Attention model saved: {ATTN_PATH}')

# Save predictions
np.save(PRED_PATH, demo_preds)
print(f'Predictions saved: {PRED_PATH}')

# Training curves plot
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle(f'Demo Training — Final: {demo_acc*100:.2f}% accuracy | {demo_auc:.4f} AUC',
             fontweight='bold')

axes[0].plot(demo_history.history['accuracy'], 'b-', linewidth=1.5)
axes[0].set_title('Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].grid(True, alpha=0.3)

axes[1].plot(demo_history.history['loss'], 'r-', linewidth=1.5)
axes[1].set_title('Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].grid(True, alpha=0.3)

axes[2].plot(demo_history.history['auc'], 'g-', linewidth=1.5)
axes[2].set_title('AUC')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('AUC')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(PLOT_PATH, dpi=150)
plt.close()
print(f'Plot saved: {PLOT_PATH}')


Attention model saved: /workspace/deepfake/preprocessed/demo_attention.keras
Predictions saved: /workspace/deepfake/preprocessed/demo_predictions.npy
Plot saved: /workspace/deepfake/preprocessed/demo_curves.png


In [9]:
# ==============================================================
# SECTION 9 — FINAL SUMMARY
# ==============================================================
print('\n' + '=' * 60)
print('TRAINING COMPLETE — SUMMARY')
print('=' * 60)
print(f'  Videos trained on : {n_videos}')
print(f'  Real              : {(demo_y==0).sum()}')
print(f'  Fake              : {(demo_y==1).sum()}')
print(f'  Final accuracy    : {demo_acc*100:.2f}%')
print(f'  Final AUC         : {demo_auc:.4f}')
print(f'  Wrong predictions : {len(demo_wrong)}')
print(f'  Training time     : {train_time/60:.1f} min')
print(f'')
print(f'  Files saved:')
print(f'    Model      : {MODEL_PATH}')
print(f'    Attention  : {ATTN_PATH}')
print(f'    Features   : {FEAT_PATH}')
print(f'    Predictions: {PRED_PATH}')
print(f'    Log        : {LOG_PATH}')
print(f'    Plot       : {PLOT_PATH}')
print(f'')
print(f'  Next step: run step3_demo')
print('=' * 60)


TRAINING COMPLETE — SUMMARY
  Videos trained on : 732
  Real              : 363
  Fake              : 369
  Final accuracy    : 100.00%
  Final AUC         : 1.0000
  Wrong predictions : 0
  Training time     : 1.3 min

  Files saved:
    Model      : /workspace/deepfake/preprocessed/demo_bilstm.keras
    Attention  : /workspace/deepfake/preprocessed/demo_attention.keras
    Features   : /workspace/deepfake/preprocessed/demo_features.npy
    Predictions: /workspace/deepfake/preprocessed/demo_predictions.npy
    Log        : /workspace/deepfake/preprocessed/demo_log.csv
    Plot       : /workspace/deepfake/preprocessed/demo_curves.png

  Next step: run step3_demo
